# Learn the definitions



### a.Tokenisation - splitting a string of text into meaningful pieces (tokens made up of words and punctuation)
- we use spaCy's tokenizer because it is built on grammar rules such as punctuation, abbreviations

In [13]:
import spacy

# a "blank" pipeline just gives us the tokenizer rules for English —
# no trained model needed for this step
nlp = spacy.blank("en")

sample_review = "Pay could be improved based off of expectations of employees, but management doesn't listen."

doc = nlp(sample_review)
tokens = [token.text for token in doc]
print(tokens)

['Pay', 'could', 'be', 'improved', 'based', 'off', 'of', 'expectations', 'of', 'employees', ',', 'but', 'management', 'does', "n't", 'listen', '.']


In [3]:
import pandas as pd

df = pd.read_csv("../data/glassdoor_reviews.csv")

# Inspect columns
print(df.columns)
df.head(10)

Index(['company_name', 'responsibility', 'date_review', 'job_title',
       'employment_details', 'location', 'overall_rating', 'work_life_balance',
       'culture_values', 'diversity_inclusion', 'career_opp', 'comp_benefits',
       'senior_mgmt', 'recommend', 'ceo_approv', 'outlook', 'headline', 'pros',
       'cons', 'source', 'X', 'X.1', 'X.2', 'X.3', 'X.4', 'X.5', 'X.6', 'X.7',
       'X.8', 'X.9', 'X.10', 'X.11', 'X.12', 'X.13', 'X.14', 'X.15', 'X.16',
       'X.17', 'X.18'],
      dtype='str')


,company_name,responsibility,date_review,job_title,employment_details,location,overall_rating,work_life_balance,culture_values,diversity_inclusion,...,X.9,X.10,X.11,X.12,X.13,X.14,X.15,X.16,X.17,X.18
0,Aesop,BCorp,2024/07/02,Retail consultant,"Current employee, more than 3 years",Sydney,3,3.0,3.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aesop,BCorp,2024/06/28,Consultant,"Former employee, less than 1 year","Denver, CO",3,4.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aesop,BCorp,2024/06/25,Consultant,"Current employee, more than 3 years",Melbourne,3,4.0,3.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aesop,BCorp,2024/06/20,People team,"Former employee, more than 3 years","London, England",3,4.0,3.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aesop,BCorp,2024/06/13,Retail assistant,"Current employee, more than 1 year",Brisbane,4,5.0,4.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Aesop,BCorp,2024/06/13,Retail consultant,Current employee,"London, England",5,5.0,5.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Aesop,BCorp,2024/06/12,Part time sales associate,Former employee,NaN,5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Aesop,BCorp,2024/06/11,Retail assistant,Former employee,Sydney,5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Aesop,BCorp,2024/06/11,Retail consultant,"Current employee, more than 1 year","Seattle, WA",5,4.0,5.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Aesop,BCorp,2024/06/05,Project manager,Former employee,Sydney,2,1.0,2.0,2.0,...,2.0,2.0,3.0,❌,⚪️,⚪️,Decent,"A relaxed and beautiful environment, an amazin...",Pay is quite low and the pressure to interact ...,https://www.glassdoor.co.uk/Reviews/Aesop-Revi...


### b. Lemmatisation - Reducing a word to it's dictionary base form
- done to make sure words in their various (noun, verb, adjectives) are not treated as unrelated which would split the data and weaken every statistic computed later
- stemming on the other hand chops off suffixes with crude rules e.g studies -> studi. It is fast but produces non-words

In [4]:
import pandas as pd
import spacy

df = pd.read_csv("../data/glassdoor_reviews.csv")
nlp = spacy.blank("en")
nlp.add_pipe("lemmatizer", config={"mode": "lookup"})
nlp.initialize()

review = df['cons'].dropna().iloc[18]
print('RAW TEXT:')
print(review)
print()

doc = nlp(review)
print('TOKEN -> LEMMA')
for token in doc:
    if not token.is_space:
        print(f'{token.text:15} -> {token.lemma_}')

RAW TEXT:
Low pay, definitely not the cost of living in any city

TOKEN -> LEMMA
Low             -> Low
pay             -> pay
,               -> ,
definitely      -> definitely
not             -> not
the             -> the
cost            -> cost
of              -> of
living          -> live
in              -> in
any             -> any
city            -> city


# step 3 - stop words and pos tagging
### stop words
- words like 'the', 'is', 'a' have verry little meaning about text and just add noise and dilute the signal.
- we remove them after tokenisation.

### part of speech tagging
- labels each token with it's grammatical role -> noun, verb, adjective
- allows and helps pull out only nouns for topic detection or only adjectives for sentiment

In [5]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")
df = pd.read_csv("../data/glassdoor_reviews.csv")

def clean_batch(texts):
    """
    Takes a list of raw strings, returns a list of cleaned strings:
    lowercased, lemmatised, with stop words and punctuation removed.
    """
    cleaned = []
    for doc in nlp.pipe(texts, batch_size=100):   # <- the key efficiency trick
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop and not token.is_punct and not token.is_space
        ]
        cleaned.append(" ".join(tokens))
    return cleaned

# fillna("") so nlp.pipe never chokes on a NaN/float
df["pros_clean"] = clean_batch(df["pros"].fillna("").tolist())
df["cons_clean"] = clean_batch(df["cons"].fillna("").tolist())

df[["pros", "pros_clean", "cons", "cons_clean"]].head()

,pros,pros_clean,cons,cons_clean
0,"good discounts, interesting coworkers, beautif...",good discount interesting coworker beautiful s...,really intense no microwaves rule which all st...,intense microwave rule store staff despise
1,Team environment is collaborative and decisive,team environment collaborative decisive,Pay could be improved based off of expectation...,pay improve base expectation employee
2,There is a lack of management and overall care...,lack management overall care high up employee ...,Rigid culture\nPretentiousness \nInconsistency...,rigid culture pretentiousness inconsistency store
3,"Great product, great brand history and ethos, ...",great product great brand history ethos genera...,"Unclear business goals, a lot of change and am...",unclear business goal lot change ambiguity res...
4,Lovely staff who are passionate and knowledgab...,lovely staff passionate knowledgable opportuni...,Despite the amount of resources they have as a...,despite resource company bother attend simple ...


## Step 4 - Named Entity Recognition
- this helps in finding specific real-world 'things' in text like people, countries, organisations, locations, dates

In [10]:
def extract_entities(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=100):
        results.append([(ent.text, ent.label_) for ent in doc.ents])
    return results

df["cons_entities"] = extract_entities(df["cons"].fillna("").tolist())
df["pros_entities"] = extract_entities(df["pros"].fillna("").tolist())
df[["cons", "cons_entities", "pros", "pros_entities"]].head(15)

,cons,cons_entities,pros,pros_entities
0,really intense no microwaves rule which all st...,[],"good discounts, interesting coworkers, beautif...",[]
1,Pay could be improved based off of expectation...,[],Team environment is collaborative and decisive,[]
2,Rigid culture\nPretentiousness \nInconsistency...,[],There is a lack of management and overall care...,"[(the last few years, DATE)]"
3,"Unclear business goals, a lot of change and am...",[],"Great product, great brand history and ethos, ...",[]
4,Despite the amount of resources they have as a...,"[(a year old, DATE)]",Lovely staff who are passionate and knowledgab...,"[(first, ORDINAL)]"
5,excellent excellent excellent excellent excellent,[],good good good good good,[]
6,No cons to mention for me.,[],"Super clean, aesthetic, modern and easy work.",[]
7,Nothing comes to my mind.,[],"Considerate brand, ongoing individual developm...",[]
8,Sometimes customers are unfamiliar with the Ae...,"[(Aesop, LOC)]",People working with you really make the work e...,[]
9,The expectations are unrealistic for the money...,[],"Great product, training team and store design.",[]


## Step 5 -  bag of words
- this converts the clean words/ vocabulary into numbers

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(df["cons_clean"])   # from your earlier cleaning step

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Matrix shape:", bow_matrix.shape)   # (num_reviews, num_unique_words)

Vocabulary size: 5823
Matrix shape: (6936, 5823)


## Step 6 - TF-IDF (Term Frequency-Inverse Document Frequency)
- Bag of words counts words and treats them all as equal.
#### TF(Term Frequency) - weighs how often a word appears in a document
#### IDF(Inverse Document Frequency) - is a penalty based on how many documents across the whole dataset contain that word. weight of rare words is pushed up while weight of common words is pushed down

- This results to common words being suppressed while rare words are amplified

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=1000)   # cap vocab size — common at scale
tfidf_matrix = tfidf_vectorizer.fit_transform(df["cons_clean"])

print("Matrix shape:", tfidf_matrix.shape)   # (num_reviews, num_features)

Matrix shape: (6936, 1000)


The 1000 features contain the words with the highest scores across the dataset. This prevents noise and adding vocabulary that wouldn't add any usefulness

## Step 7 - Word Embeddings (Word2Vec)
- Bag of words and TFIDF have words in their own isolated column and the model here would not have a concept of a word meaning.
- Word2Vec learns the dense vector for each word and look at which words tend to appear near each other across a huge amount of text.
- Words with similar context end up close to each other in the vector space.

In [16]:
from gensim.models import Word2Vec

# convert the tokenized text column into a list of tokens stored in a sentences variable that is then used a parameter by the embedding model
sentences = df["cons_clean"].apply(lambda x: x.split()).tolist()

# train the model
model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=1, seed=42)

print(model.wv.most_similar("management", topn=3))


[('level', 0.9930779933929443), ('communication', 0.9927992224693298), ('leadership', 0.992408037185669)]


When converting the words to vectors, covert the cleaned tokenised column into a list of words first, it is also important to split each inner string in the list of words (a list where each element is itself a list of individual words)

## Step 8 _ Classification
### 8a. Turning ratings into a label
-This is done because we do not have a positive/relative column, just an overall_rating column
- Treat ratings between 1 & 2 as negative and those between 4 & 5 as positive, drop the 3 becuase it is average/ it is fuzzy and doesn't help the model make good predictions

In [6]:
# drop rows where we don't have the text or the rating — can't use them either way
df = df.dropna(subset=["pros", "cons", "overall_rating"])

# drop the ambiguous middle rating
df = df[df["overall_rating"] != 3]

# create a new column 'label' with 1 = positive (rating 4 or 5), 0 = negative (rating 1 or 2)
df["label"] = (df["overall_rating"] >= 4).astype(int)

print(df["label"].value_counts())
print()
print(df[["overall_rating", "label"]].head(10))

label
1    4206
0    1422
Name: count, dtype: int64

    overall_rating  label
4                4      1
5                5      1
6                5      1
7                5      1
8                5      1
9                2      0
10               2      0
13               2      0
14               2      0
15               5      1


### 8b. Combining and cleaning the text
- We combine the pros and cons to give the model richer text to work with per review

In [8]:
# combine pros + cons into a single text field per row
df["full_text"] = df["pros"].astype(str) + " " + df["cons"].astype(str)

print(df[["pros", "cons", "full_text"]].head(3))

                                                pros  \
4  Lovely staff who are passionate and knowledgab...   
5                           good good good good good   
6      Super clean, aesthetic, modern and easy work.   

                                                cons  \
4  Despite the amount of resources they have as a...   
5  excellent excellent excellent excellent excellent   
6                         No cons to mention for me.   

                                           full_text  
4  Lovely staff who are passionate and knowledgab...  
5  good good good good good excellent excellent e...  
6  Super clean, aesthetic, modern and easy work. ...  


In [9]:
# clean the full_text column

def clean_batch(texts):
    cleaned = []
    for doc in nlp.pipe(texts, batch_size=200):
        tokens = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop]
        cleaned.append(" ".join(tokens))
    return cleaned

df["clean_text"] = clean_batch(df["full_text"].tolist())

The t.is_alpha is used here to strip out numbers, punctuation and symbols in one go.

In [10]:
print(df[["full_text", "clean_text"]].head(3))


                                           full_text  \
4  Lovely staff who are passionate and knowledgab...   
5  good good good good good excellent excellent e...   
6  Super clean, aesthetic, modern and easy work. ...   

                                          clean_text  
4  lovely staff passionate knowledgable opportuni...  
5  good good good good good excellent excellent e...  
6  super clean aesthetic modern easy work con men...  


### 8c. Train, test, split

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 4502
Test size: 1126


- The function takes the X (text) and y (labels).
- Random_state makes the results reproducible each time we run the code we get consistent results making it easier to debug.
- Stratify df["label"] takes care of the class imbalance, randomly splitting could put a few negative values to the training set since there is data imbalance which would skew the whole process.

The model therefore is going to be better at recognising positive reviews because they are the majority due to the class imbalance.

## 8d. TF-IDF vectorising the train and test sets

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000)

X_train_vec = vectorizer.fit_transform(X_train)   # fit AND transform
X_test_vec = vectorizer.transform(X_test)          # transform ONLY

**fit** enables the vectorizer to build its vocabulary. It scans the text given, decides which words matter based on the max_features parameter and calculates the IDF for each word across the documents it saw/learned from.

**transform** applies an already-built vocabulary and weighting scheme to turn text into numbers. It doesn't learn anything new rather it just looks up wors it already knows.

In [24]:
print("X_train_vec shape:", X_train_vec.shape)
print("X_test_vec shape: ", X_test_vec.shape)
print()
print("Vocabulary learned FROM TRAINING ONLY, size:", len(vectorizer.vocabulary_))
print()
print("Sample of learned vocabulary words:", list(vectorizer.vocabulary_.keys())[:10])

X_train_vec shape: (4502, 3000)
X_test_vec shape:  (1126, 3000)

Vocabulary learned FROM TRAINING ONLY, size: 3000

Sample of learned vocabulary words: ['awesome', 'job', 'good', 'con', 'teach', 'work', 'fit', 'day', 'schedule', 'fast']


## 8e. Train the classifier
- We hand the model the numeric vectors and their correct labels and let it learn the relationship between them

In [27]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, class_weight="balanced") # treat the mistakes on the minority side as more costly
clf.fit(X_train_vec, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## 8f. Model Evaluation

In [28]:
from sklearn.metrics import accuracy_score, classification_report

preds = clf.predict(X_test_vec)

print("Accuracy:", round(accuracy_score(y_test, preds), 3))
print()
print(classification_report(y_test, preds, target_names=["negative", "positive"]))

Accuracy: 0.889

              precision    recall  f1-score   support

    negative       0.75      0.84      0.79       285
    positive       0.94      0.91      0.92       841

    accuracy                           0.89      1126
   macro avg       0.85      0.87      0.86      1126
weighted avg       0.89      0.89      0.89      1126



The recall for positive is 98% meaning h=the model almost finds all the positive reviews, 60% the model finds the negatvie reviews.

The model's precision on the negative values is 0.91 meaning it is right 91% of the time.

Currently the model's accuracy is at 0.88.

### 9a. From word vectors to a document vector
- A classifer needs one fixed-length vector per review to make sure there is uniformity for the classifier.
- Some words, in this case reviews have different number of words, averaging all words in a review together makes them loose word orer and are all treated equally.
- Averaging all the words results into a dimensional vector that represents the whole review by using the average.

In [13]:
from gensim.models import Word2Vec
import numpy as np

# train Word2Vec on the TRAINING split only (X_train from our earlier split)
train_sentences = [text.split() for text in X_train]

w2v_model = Word2Vec(train_sentences, vector_size=100, window=5, min_count=2, workers=2, seed=42, epochs=15)

print("Vocabulary size:", len(w2v_model.wv.index_to_key))

def document_vector(text, model):
    """
    Average the word vectors for every word in a document.
    Words not in the vocabulary are simply skipped.
    """
    words = text.split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)   # fallback for empty/unknown text
    return np.mean(vectors, axis=0)

# quick sanity check on one review
sample_vec = document_vector(X_train.iloc[0], w2v_model)
print("Sample review:", X_train.iloc[0])
print("Resulting vector shape:", sample_vec.shape)
print("First 5 values:", sample_vec[:5])

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary size: 5243
Sample review: awesome job good con good job
Resulting vector shape: (100,)
First 5 values: [ 0.7438124   0.3120254  -0.5294996   0.6869561  -0.36589396]


### 9b. Building the full feature matrices and training the classifier
- We do this by applying document_vector() to every review both in the train and test sets turning eah into a 100-dimensional row

In [14]:
import numpy as np

# apply document_vector() to every review — building one row per document
X_train_w2v = np.array([document_vector(text, w2v_model) for text in X_train])
X_test_w2v = np.array([document_vector(text, w2v_model) for text in X_test])

print("X_train_w2v shape:", X_train_w2v.shape)
print("X_test_w2v shape: ", X_test_w2v.shape)

X_train_w2v shape: (4502, 100)
X_test_w2v shape:  (1126, 100)


- The shape is now 100 columns regardless of vocabulary size, previously the columns were 3000 which represented each word in the vocabulary.
- This is advantagious because we have a smaller denser feature representation that takes up less space and will be faster to train.

In [15]:
# train the classifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train)

preds_w2v = clf_w2v.predict(X_test_w2v)

print("Accuracy:", round(accuracy_score(y_test, preds_w2v), 3))
print()
print(classification_report(y_test, preds_w2v, target_names=["negative", "positive"]))

Accuracy: 0.854

              precision    recall  f1-score   support

    negative       0.80      0.56      0.66       285
    positive       0.87      0.95      0.91       841

    accuracy                           0.85      1126
   macro avg       0.83      0.76      0.78      1126
weighted avg       0.85      0.85      0.85      1126



- The TF-IDF performed better compared to the averaged Word2Vec due to reasons such as:
- Using the Word2Vec required word averaging which makes the words loose meaning that was used by the TF-IDF.
- Word2Vec requires many words to learn the meaning but we use 4,502 training reviews which in this case is a small number.
- TF-IDF is a strong baseline because it amplifies keywords while Word2Vec captures nuances semantic similarity which isn't as important in this task.